# 지도학습 실습

**Supervised Learning**

입력과 정답이 짝지어진 데이터로 입력과 결과의 관계를 학습하는 방법.

소재 분야에서 이해하기: 조성별로 측정한 전도도를 정답으로 사용한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Google ML 용어집](https://developers.google.com/machine-learning/glossary)

## 1. 정답이 있는 데이터

지도학습은 입력과 정답의 짝이 필요합니다. 여기서는 공정 조건이 입력, 경도가 정답입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
import pandas as pd

table = pd.DataFrame(X, columns=FEATURES)
table['경도(정답)'] = y
print(table.head())

## 2. 정답의 양이 성능을 바꿉니다

정답을 확보한 시료 수를 늘려가며 시험 오차를 봅니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
sizes, scores = [], []
for n in (10, 20, 40, 80, 120, len(X_train)):
    model = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_train[:n], y_train[:n])
    sizes.append(n); scores.append(mean_absolute_error(y_test, model.predict(X_test)))
    print('정답 %3d개 -> MAE %.2f HV' % (n, scores[-1]))

plt.plot(sizes, scores, 'o-'); plt.xlabel('labelled samples'); plt.ylabel('test MAE'); plt.show()

## 3. 해석

정답이 늘면 오차가 줄지만 어느 지점부터는 완만해집니다. 측정을 더 할지, 모델을 바꿀지
판단할 때 이 곡선을 근거로 쓸 수 있습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#supervised)을 여세요.